# Fresh native option pricing workshop

Retain the original five-bit, 32-point asset grid, log-normal mu=0.7 and
sigma=0.13, strike1.9, three-standard-deviation bounds, normalized PDF,
in-the-money threshold, square-root payoff amplitude, and source width budget
28. Distribution, payoff table and expected value are recomputed from these
parameters. No frozen Qmod or rounded quantum-counterpart tables supply the model data.

The existing native qfunc option loader and reflection implement the quantum
family. A bounded local iterative amplitude estimator requests only sampled
indicator counts, with epsilon0.05 and alpha0.01. It uses simultaneous Hoeffding
intervals, at most64adaptive steps, k<=16 and256fresh samples per batch. It is an
explicit local IQAE profile, not the proprietary Classiq implementation. Each
new structural k is synthesized and executed once, then its exact local marginal
is reused for fresh binomial draws. This is not runtime parameter binding or a
hardware timing model. Independent Circuit and direct-vector references check
full absolute six-qubit states and named indicator values before readout.

The asset remains a private possibly entangled environment after drop; it is not
clean compiler workspace. Exact classical expected payoff is validation-only,
never an input to adaptive power selection or success counts. Source implicit
sampling defaults are replaced by explicit recorded local budgets and seed6101.
Final payoff validation retains the source tolerance factor10. No remote
job, new SciPy dependency, hardware, pricing advantage or scalability claim.


In [ ]:
from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.algorithms.amplitude_amplification_and_estimation.quantum_counting.overview.iterative import (
    estimate,
)
from examples.synthesis.applications.finance.option_pricing.overview.model import (
    fresh_model,
)
from examples.synthesis.applications.finance.option_pricing.overview.plots import (
    plot_distribution,
)
from examples.synthesis.applications.finance.option_pricing.overview.workflow import (
    IndicatorSampler,
)

num_qubits = 5
mu, sigma, strike = 0.7, 0.13, 1.9
max_width = 28
seed = 6101
print("Original model parameters:", num_qubits, mu, sigma, strike)


In [ ]:
model = fresh_model(mu, sigma, strike)
print("Fresh grid bounds:", min(model.grid), max(model.grid))
print("Fresh normalized probabilities:", model.probabilities)


In [ ]:
assert len(model.grid) == len(model.probabilities) == 32
assert abs(sum(model.probabilities) - 1) < 1e-14
figure_path = plot_distribution(
    model, Path("artifacts/development/option_pricing_workshop")
)
display(Image(filename=str(figure_path)))


In [ ]:
print("Fresh exact loader normalization:", sum(model.probabilities))
print("The native root uses prepare(probabilities, angles, asset, indicator)")
print("It composes inplace_prepare_state and checked payoff rotations")


In [ ]:
grid_step = (max(model.grid) - min(model.grid)) / 31
print("In-the-money encoded threshold:", model.cutoff)
print("Original price/word conversion step:", grid_step)
assert all(
    (word >= model.cutoff) == (price >= strike) for word, price in enumerate(model.grid)
)


In [ ]:
scaling_factor = model.scale
print("Payoff scaling factor:", scaling_factor)
print("Fresh effective payoff rotation table:", model.angles)
print(
    "Below-strike rows remain identity; above-strike probability equals payoff/scaling_factor"
)


In [ ]:
sampler = IndicatorSampler(model, seed=seed, max_width=max_width)
print("Native bounded family width budget:", max_width)
print(
    "Count-only iterative profile: epsilon=0.05, alpha=0.01, k<=16, batch256, steps<=64"
)


In [ ]:
program = sampler.ensure(0)
print("Actual base resources:", program.report["resources"])
print("Named output:", program.registers)


In [ ]:
estimated = estimate(sampler, epsilon=0.05, alpha=0.01)
for iteration in estimated.iterations:
    print(iteration)
print("Saved actual native family programs:", tuple(sampler.programs))
print("Independent complete-state checks:", tuple(sampler.checks.values()))


In [ ]:
measured_payoff = estimated.estimate * scaling_factor
confidence_interval = tuple(
    value * scaling_factor for value in estimated.confidence_interval
)
print("Measured payoff:", measured_payoff)
print("Scaled price confidence interval:", confidence_interval)


In [ ]:
expected_payoff = sum(
    max(price - strike, 0) * probability
    for price, probability in zip(model.grid, model.probabilities, strict=True)
)
print("Independent classical expected payoff:", expected_payoff)


In [ ]:
source_tolerance = 10 * (confidence_interval[1] - confidence_interval[0])
assert abs(measured_payoff - expected_payoff) <= source_tolerance
assert estimated.confidence_interval[1] - estimated.confidence_interval[0] <= 0.1
print("Original source tolerance passed:", source_tolerance)
print("Total actual adaptive samples:", sum(row.shots for row in estimated.iterations))
